## Preparando el ambiente
Instalamos las librerías y configuramos nuestra API Key

In [2]:
import numpy as np
import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.messages import HumanMessage, SystemMessage
import os
import ast
from elasticsearch import Elasticsearch

In [64]:
from huggingface_hub import login
HF_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
login(HF_API_TOKEN)

# Prueba sin RAG
Llamamos el LLM LLama sin ninguna modificación y vemos que alucina o bien desconoce esta información

In [ ]:
# Step 1: Define HuggingFaceEndpoint
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-70B-Instruct",
    task="conversational", #THIS MODEL ONLY WORKS WITH CONVERSATIONAL TASK
    temperature=0.7,
    max_new_tokens=512,
)

# Step 2: Initialize ChatHuggingFace
chat = ChatHuggingFace(llm=llm)

# Step 3: Define prompt and parser
prompt = ChatPromptTemplate.from_template("Responde en español: {question}")
parser = StrOutputParser()

# Step 4: Build chain
chain = prompt | chat | parser

In [ ]:
response = chain.invoke({"question": "¿Dame mas detalles sobre el destornillador inalambrico de la marca Truper?"})
print(response)

Claro! El destornillador inalámbrico de la marca Truper es una herramienta eléctrica portátil y versátil que te permite realizar various tareas de atornillado y desatornillado con facilidad y comodidad. A continuación, te proporciono algunos detalles clave sobre este producto:

**Características**

* Potente motor inalámbrico que ofrece una mayor libertad de movimiento y versatilidad en el trabajo.
* Batería recargable de iones de litio que proporciona una larga duración de vida útil y rápida recarga.
* Velocidad variable que se ajusta a las necesidades del trabajo, con una velocidad máxima de 1.800 rpm.
* Par de torsión máximo de 130 Nm, suficiente para manejar tareas de atornillado y desatornillado de medianas a pesadas.
* Mango ergonómico diseñado para una mayor comodidad y control durante el uso.

**Accesorios y opciones**

* Viene con una variedad de brocas y bits para diferentes tipos de tornillos y tareas, incluyendo brocas hexagonales, Phillips, flathead, etc.
* Opcionalmente, 

# Configurando el RAG

In [3]:
# WE GET THE DESCRIPTIONS FOR ALL PRODUCTS
import ast 
electric_tool_truper = pd.read_csv("../notebooks/data/sample_electric_truper_products.csv", sep=";")
electric_tool_truper_des = electric_tool_truper["Descripción"].apply(lambda x: ast.literal_eval(x)["descripcion"]).tolist()
df_desc = pd.DataFrame(electric_tool_truper_des, columns=['Text'])
df_desc

,Text
0,"Marca Truper, Línea 18153, Modelo 18153, Tipo ..."
1,"Marca Truper, Modelo LLM-6L, Tipo de producto ..."
2,"Marca Truper, Modelo Torx-7l, Tipo de llave Co..."
3,"Marca Truper, Modelo MAND-7/16, Tipo de mandri..."
4,"Marca Truper, Modelo PICA-X, Tipo de producto ..."
...,...
89,"Marca Truper, Modelo 14182 ""Grata De Copa 3""""..."
90,"Marca Truper, Modelo ALLX-7M, Tipo de llave Al..."
91,"""Marca Truper, Modelo PPC-11R, Tipo de punta C..."
92,"""Marca Truper, Modelo JOY-6, Formato de venta ..."


In [5]:
#In the case where we get more information about a product for now we will embed each paragraph separately

# electric_tool_truper_descriptions = electric_tool_truper_descriptions.apply(lambda x: x.replace("\n", " "))

# electric_tools_document = " ".join(electric_tool_truper_descriptions.tolist())
# document = Document(page_content=electric_tools_document, metadata={"source": "electric_tool_truper"})

"""
## 1. Diviviendo los documentos en chunks
 Los documentos son divididos en pedacitos, o chunks, para que puedan ser correctamente procesados en este contexto.

"""
# text_splitter = RecursiveCharacterTextSplitter(
#                   chunk_size=450,
#                   chunk_overlap=0)


# splits = text_splitter.split_documents(docs)

# for index, split in enumerate(splits):
#   print(f"SPLIT {index + 1}")
#   print(split.page_content)
#   print("--")

# Next the idea will be to save them on a vectorial database

'\n## 1. Diviviendo los documentos en chunks\n Los documentos son divididos en pedacitos, o chunks, para que puedan ser correctamente procesados en este contexto.\n\n'

## Generando los embeddings
Los embeddings son representaciones númericas de la información. Existen varios proveedores de Embeddings, pero vamos a usar los de Hugging Face de la libreria de SentenceTransformer

In [ ]:
#ok
# # Crearemos una función que luego aplicaremos a cada texto
# embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# def embeddings_fn(text):
#   """
#   Crea los embeddings dado un texto
#   """
#   return embedder.encode(text)

In [ ]:
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
es = Elasticsearch("http://localhost:9200")
index_name = "ferriatienda-electrical-products-sample"

In [46]:
# Calculamos los embeddings para cada texto en nuestro DataFrame
df_desc['Embeddings'] = df_desc['Text'].apply(embeddings_fn)
df_desc.head(10)

,Text,Embeddings
0,"Marca Truper, Línea 18153, Modelo 18153, Tipo ...","[-0.107427835, 0.058696587, -0.065981224, -0.1..."
1,"Marca Truper, Modelo LLM-6L, Tipo de producto ...","[-0.0062168343, 0.084737554, 0.005445822, 0.07..."
2,"Marca Truper, Modelo Torx-7l, Tipo de llave Co...","[-0.08590685, -0.009702208, 0.019668588, 0.118..."
3,"Marca Truper, Modelo MAND-7/16, Tipo de mandri...","[-0.0038052632, 0.074735135, -0.09987003, -0.1..."
4,"Marca Truper, Modelo PICA-X, Tipo de producto ...","[-0.14511925, 0.11315375, 0.074952796, -0.0158..."
5,"Marca Truper, Número de pieza 2.0, Tipo de veh...","[-0.02766481, 0.0760282, 0.12120334, 0.0438391..."
6,"Marca Truper, Modelo C-3/4X10, Tipo de herrami...","[-0.018527979, -0.03459798, 0.0756945, -0.1341..."
7,"Marca Truper, Modelo C-5/8X8, Formas de las ho...","[-0.18228476, 0.10051693, -0.04590613, 0.03744..."
8,"Marca Truper, Modelo 12112 CINCEL CORTE FRIO ...","[-0.029566733, 0.073624976, 0.13395947, -0.033..."
9,"Marca Truper, Modelo CEA-10, Material de las c...","[-0.056360785, 0.04847854, 0.0011040487, -0.02..."


¿Cuántas dimensiones tiene cada vector?

In [47]:
len(df_desc['Embeddings'][0])

384

## Buscando el chunk más relevante

In [ ]:
#With Elasticsearch
def find_best_description(query: str):
    query_vec = model.encode(query).tolist()
    
    response = es.search(index=index_name, knn={"field": "embedding", "k": 1, "num_candidates": 5, "query_vector": query_vec})
    hits = response["hits"]["hits"]
    
    if hits:
        return hits[0]["_source"]["text"]
    else:
        return "No tengo suficiente información para responder eso."

In [ ]:
#OK
# def find_best_description(query, dataframe, embedder):
#     query_embedding = embedder.encode([query])
#     doc_embeddings = np.stack(dataframe["Embeddings"].to_numpy())
#     similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
#     best_idx = np.argmax(similarities)
#     return dataframe.iloc[best_idx]["Text"]

Notese que nuestro ejercicio entrega sólo un resultado. Sin embargo, veremos ejemplos donde se entregan más resultados de vuelta, por ejemplo los mejores 4.

In [49]:
query = "Dame informacion sobre el destornillador eléctrico inalámbrico de la marca Truper"
best_passage = find_best_description(query, df_desc, embedder)
best_passage

'Marca Truper, Línea 18153, Modelo 18153, Tipo de producto Destornillador, Tipo de destornillador eléctrico Compacto, Es inalámbrico Sí, Tamaño del mandril 10 mm, Encastre 3/8, Torque máximo 22 Nm, Velocidad mínima de rotación 350 rpm, Velocidad máxima de rotación 1.200 rpm, Accesorios incluidos Batería  Diseño ligero para mayor comodidadDoble engranaje con selector de 2 velocidades mecánicas, botón de dirección de giro y bloqueo del interruptorBroquero de cambio rápido con seguro de retenciónLuz LED para iluminar área de trabajo Indicador de nivel de carga de batería.'

## Generando la respuesta con un LLM

In [53]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    Responde a la pregunta del usuario usando únicamente el texto de referencia. 
    No inventes información. 
    Si no encuentras la respuesta en el texto, responde "No tengo suficiente información para responder eso".

    Texto de referencia:
    {relevant_passage}

    Pregunta:
    {query}

    Respuesta:
    """)
])
parser = StrOutputParser()

chain = prompt | chat | parser



In [ ]:
# # WITHOUT THE CHAT AT THE ENDPOINT
# def answer(query):
#     context = find_best_description(query, df_desc, embedder)
#     return chain.invoke({"query": query, "relevant_passage": context})

# def answer(query):
#     context = find_best_description(query, df_desc, embedder)
#     prompt_value = prompt.invoke({"query": query, "relevant_passage": context})
#     raw_output = llm.invoke(prompt_value.to_string())
#     return output_parser.invoke(raw_output)

In [ ]:
chat_history = []

def answer(query):
    # Step 1: Get relevant context
    context = find_best_description(query ) #with elasticsearch , df_desc, embedder)

    # Step 2: Build the prompt in conversational format
    system_message = SystemMessage(content=(
        "Eres un asistente experto en herramientas eléctricas en una Ferretería. "
        "Responde en español de forma clara y concisa usando solo el contexto provisto. "
        "Si la información no está en el texto, responde: 'No tengo suficiente información para responder eso'."
    ))

    user_message = HumanMessage(content=(
        f"""
        Texto de referencia: {context}

        Pregunta: {query}
        """
    ))

    # Step 3: Update chat history
    chat_history.extend([system_message, user_message])

    # Step 4: Get response from LLM
    response = chat.invoke(chat_history)

    # Step 5: Optionally clear history if not continuing
    chat_history.clear()

    return response.content

In [58]:
consulta = "Dame informacion sobre el destornillador eléctrico inalámbrico de la marca Truper"
respuesta = answer(consulta)
print("🧠 Pregunta:", consulta)
print("💬 Respuesta:", respuesta)
#mejorar aun mas pero pues prometedor 

🧠 Pregunta: Dame informacion sobre el destornillador eléctrico inalámbrico de la marca Truper
💬 Respuesta: Según la información proporcionada, el destornillador eléctrico inalámbrico de la marca Truper tiene las siguientes características:

* Es compacto y ligero para mayor comodidad
* Tiene un tamaño de mandril de 10 mm y un encastre de 3/8
* Tiene un torque máximo de 22 Nm
* La velocidad de rotación varía entre 350 rpm (mínima) y 1.200 rpm (máxima)
* Tiene un doble engranaje con selector de 2 velocidades mecánicas
* Incluye un botón de dirección de giro y bloqueo del interruptor
* Cuenta con un broquero de cambio rápido con seguro de retención
* Tiene una luz LED para iluminar el área de trabajo
* Incluye un indicador de nivel de carga de batería
* Viene con una batería incluida


OTHER OPTIONS : 

In [61]:
# ### OK WORKS 
# # 1. Créer l'endpoint HuggingFace
# llm = HuggingFaceEndpoint(
#     repo_id="meta-llama/Meta-Llama-3-70B-Instruct",
#     task="conversational",
#     temperature=0.7,
#     max_new_tokens=512,
# )

# # 2. Créer un modèle de chat à partir du llm
# chat = ChatHuggingFace(llm=llm)

# # 3. Définir un prompt
# prompt = ChatPromptTemplate.from_template("Responde en español: {question}")
# parser = StrOutputParser()

# # 4. Créer une chaîne LangChain
# chain = prompt | chat | parser

# # 5. Appeler la chaîne
# response = chain.invoke({"question": "¿Para que sirve un destornillador?"})
# print(response)

In [62]:
# import pandas as pd
# import numpy as np
# import ast
# from sklearn.metrics.pairwise import cosine_similarity
# from sentence_transformers import SentenceTransformer
# from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser

# # ---------------------------
# # 1. Load and prepare your data
# # ---------------------------
# electric_tool_truper = pd.read_csv("../notebooks/data/sample_electric_truper_products.csv", sep=";")
# electric_tool_truper_des = electric_tool_truper["Descripción"].apply(lambda x: ast.literal_eval(x)["descripcion"]).tolist()
# df_desc = pd.DataFrame(electric_tool_truper_des, columns=['Text'])

# embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# def embeddings_fn(text):
#     return embedder.encode(text)

# df_desc['Embeddings'] = df_desc['Text'].apply(embeddings_fn)

# # ---------------------------
# # 2. Define the retriever
# # ---------------------------
# def find_best_description(query, dataframe, embedder):
#     query_embedding = embedder.encode([query])
#     doc_embeddings = np.stack(dataframe["Embeddings"].to_numpy())
#     similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
#     best_idx = np.argmax(similarities)
#     return dataframe.iloc[best_idx]["Text"]

# # ---------------------------
# # 3. Define the LLM
# # ---------------------------
# llm = HuggingFaceEndpoint(
#     repo_id="meta-llama/Meta-Llama-3-70B-Instruct",
#     task="conversational",
#     temperature=0.7,
#     max_new_tokens=512,
# )

# chat = ChatHuggingFace(llm=llm)

# # ---------------------------
# # 4. Define LangChain prompt + parser
# # ---------------------------
# prompt = ChatPromptTemplate.from_messages([
#     ("system", """
#     Responde a la pregunta del usuario usando únicamente el texto de referencia. 
#     No inventes información. 
#     Si no encuentras la respuesta en el texto, responde "No tengo suficiente información para responder eso".

#     Texto de referencia:
#     {relevant_passage}

#     Pregunta:
#     {query}

#     Respuesta:
#     """)
# ])
# parser = StrOutputParser()

# chain = prompt | chat | parser

# # ---------------------------
# # 5. Define the final RAG function
# # ---------------------------
# def answer(query):
#     context = find_best_description(query, df_desc, embedder)
#     return chain.invoke({"query": query, "relevant_passage": context})

# # ---------------------------
# # 6. Test it
# # ---------------------------
# consulta = "Dame informacion sobre el destornillador eléctrico inalámbrico de la marca Truper"
# respuesta = answer(consulta)
# print("🧠 Pregunta:", consulta)
# print("💬 Respuesta:", respuesta)